In [3]:
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))

def build_model(architecture, num_classes):
    if architecture == 'resnet18':
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif architecture == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif architecture == 'mobilenet_v2':
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f'Unknown architecture: {architecture}')
    return model.to(device)

In [4]:
try:
    from thop import profile
except ImportError:
    print("Installing thop...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "thop", "-q"])
    from thop import profile

import pandas as pd
import torch

print("="*60)
print("FLOPs Calculation for ResNet-18")
print("="*60)

flops_results = []
dataset_configs = [
    ("miracle9to9", 2),
    ("malaria", 4),
    ("iml_malaria", 4),
]

for dataset_name, num_classes in dataset_configs:
    # Use the project's build_model function to match your training exactly
    model = build_model("resnet18", num_classes)
    model.to(device)
    model.eval()
    
    # ResNet-18 was trained on 128x128 images in your notebook
    dummy_input = torch.randn(1, 3, 128, 128).to(device)
    
    # Calculate MACs and parameters
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    
    # Multiply MACs by 2 to get true FLOPs
    flops = 2 * macs
    
    flops_results.append({
        "Dataset": dataset_name,
        "Num Classes": num_classes,
        "FLOPs (M)": flops / 1e6,
        "Parameters (M)": params / 1e6,
    })
    
    print(f"\n{dataset_name} (num_classes={num_classes}):")
    print(f"  FLOPs: {flops / 1e6:.2f}M")
    print(f"  Parameters: {params / 1e6:.2f}M")

# Display as a pretty HTML table
flops_df = pd.DataFrame(flops_results)
print("\n" + "="*60)
print("FLOPs Summary Table")
print("="*60)
display(flops_df)

FLOPs Calculation for ResNet-18

miracle9to9 (num_classes=2):
  FLOPs: 1190.87M
  Parameters: 11.18M

malaria (num_classes=4):
  FLOPs: 1190.88M
  Parameters: 11.18M

iml_malaria (num_classes=4):
  FLOPs: 1190.88M
  Parameters: 11.18M

FLOPs Summary Table


,Dataset,Num Classes,FLOPs (M),Parameters (M)
0,miracle9to9,2,1190.874112,11.177538
1,malaria,4,1190.876160,11.178564
2,iml_malaria,4,1190.876160,11.178564
